# Solutions · 04 · Interpolation: reading between the lines of a table

Worked solutions to the exercises of [notebook 04](../notebooks/04_interpolation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import interpolate, roots
def p_antoine(T):          # kPa
    return 10**(8.07131 - 1730.63/(233.426 + T)) * 0.133322

## Exercise 1

Repeat the error study with the table every 20 °C. How does the maximum error of each method
   scale with the spacing? (Theory: linear $\propto h^2$, cubic spline $\propto h^4$.)

In [2]:
T_fine = np.linspace(20, 100, 800)
p_fine = p_antoine(T_fine)
interior = (T_fine > 40) & (T_fine < 80)
print(f"{'spacing':>8}{'linear':>10}{'spline (all)':>14}{'spline (40-80 °C)':>19}")
res = []
for dT in (20, 10, 5):
    Tt = np.arange(20.0, 100.1, dT)
    e_lin = np.max(np.abs(interpolate.linear(Tt, p_antoine(Tt), T_fine)/p_fine - 1))
    s = interpolate.CubicSpline(Tt, p_antoine(Tt))(T_fine)/p_fine - 1
    res.append((e_lin, np.max(np.abs(s)), np.max(np.abs(s[interior]))))
    print(f"{dT:>6} °C{res[-1][0]:>10.2e}{res[-1][1]:>14.2e}{res[-1][2]:>19.2e}")
for i in range(2):
    print(f"halving the spacing reduces the errors by: linear {res[i][0]/res[i+1][0]:.1f}x, "
          f"spline {res[i][1]/res[i+1][1]:.1f}x, spline interior {res[i][2]/res[i+1][2]:.1f}x")

 spacing    linear  spline (all)  spline (40-80 °C)
    20 °C  1.52e-01      3.06e-02           1.71e-02
    10 °C  3.93e-02      1.30e-02           9.29e-04
     5 °C  1.01e-02      3.66e-03           1.46e-05
halving the spacing reduces the errors by: linear 3.9x, spline 2.4x, spline interior 18.4x
halving the spacing reduces the errors by: linear 3.9x, spline 3.5x, spline interior 63.8x


Linear interpolation behaves as theory predicts: halving the spacing cuts the error about 4x ($h^2$). The
natural spline falls short of the promised 16x ($h^4$) over the whole table, because its **end condition**
(zero curvature at both ends) is wrong for this curve, which is strongly curved at 100 °C. That error is
largest near the ends and converges only like $h^2$; away from the ends the spline does much better.
Exercise 4 removes the problem with a better end condition.

## Exercise 2

Use a spline of $\ln p$ against $T$ to find the boiling temperature at 50 kPa (combine with a
   root finder from notebook 02). Compare with the Antoine equation solved directly.

In [3]:
Tt = np.arange(10.0, 101.0, 10.0)
s = interpolate.CubicSpline(Tt, np.log(p_antoine(Tt)))
T_boil = roots.brent(lambda T: s(T) - np.log(50.0), 10, 100).root
T_exact = 1730.63/(8.07131 - np.log10(50.0/0.133322)) - 233.426
print(f"spline of ln p: {T_boil:.4f} °C;  Antoine equation inverted: {T_exact:.4f} °C")

spline of ln p: 81.3900 °C;  Antoine equation inverted: 81.3919 °C


The boiling point at 50 kPa follows from a root search on the interpolant: about 81.4 °C, agreeing with
the Antoine equation (inverted exactly) to a few thousandths of a degree. Interpolating $\ln p$ makes the
curve nearly linear, which is why so few table points suffice.

## Exercise 3

Replace the equally spaced points in the Runge example by Chebyshev points
   $x_j = \cos(j\pi/n)$. What happens to the polynomial?

In [4]:
f = lambda x: 1/(1 + 25*x**2)
xq = np.linspace(-1, 1, 1000)
for n in (10, 20, 40):
    x_eq = np.linspace(-1, 1, n + 1)
    x_ch = np.cos(np.arange(n + 1) * np.pi / n)
    e_eq = np.max(np.abs(interpolate.lagrange(x_eq, f(x_eq), xq) - f(xq)))
    e_ch = np.max(np.abs(interpolate.lagrange(x_ch, f(x_ch), xq) - f(xq)))
    print(f"degree {n:>2}: equally spaced error {e_eq:9.2e}   Chebyshev points error {e_ch:.2e}")

degree 10: equally spaced error  1.92e+00   Chebyshev points error 1.32e-01
degree 20: equally spaced error  5.98e+01   Chebyshev points error 1.77e-02
degree 40: equally spaced error  1.04e+05   Chebyshev points error 3.40e-04


With Chebyshev points - clustered towards the ends of the interval - the polynomial error **decreases**
steadily with the degree, while with equally spaced points it explodes. Runge's phenomenon is caused by
the choice of points, not by polynomials themselves; Chebyshev-based methods are the basis of highly
accurate numerical libraries.

## Exercise 4

**Implement it yourself:** turn `spline_by_hand` into a *clamped* spline with prescribed end slopes $y'_0$ and $y'_n$ (only the first and last rows of the system change - derive them). Use slopes from the Antoine equation: does the error near the ends of the table fall?

In [5]:
def spline_clamped(x, y, xq, d0, dn):
    h = np.diff(x); n = len(x)
    A, rhs = np.zeros((n, n)), np.zeros(n)
    A[0, 0], A[0, 1] = 2*h[0], h[0]                       # clamped start: slope d0
    rhs[0] = 6*((y[1] - y[0])/h[0] - d0)
    A[-1, -2], A[-1, -1] = h[-1], 2*h[-1]                 # clamped end: slope dn
    rhs[-1] = 6*(dn - (y[-1] - y[-2])/h[-1])
    for i in range(1, n - 1):
        A[i, i-1], A[i, i], A[i, i+1] = h[i-1], 2*(h[i-1] + h[i]), h[i]
        rhs[i] = 6*((y[i+1] - y[i])/h[i] - (y[i] - y[i-1])/h[i-1])
    M = np.linalg.solve(A, rhs)
    i = np.clip(np.searchsorted(x, xq) - 1, 0, n - 2)
    a, b, hi = x[i+1] - xq, xq - x[i], h[i]
    return (M[i]*a**3 + M[i+1]*b**3)/(6*hi) + (y[i]/hi - M[i]*hi/6)*a + (y[i+1]/hi - M[i+1]*hi/6)*b

dpdT = lambda T: p_antoine(T) * np.log(10) * 1730.63 / (233.426 + T)**2
Tt = np.arange(10.0, 101.0, 10.0); pt = p_antoine(Tt)
T_fine = np.linspace(10, 100, 900); p_fine = p_antoine(T_fine)
e_nat = np.abs(interpolate.CubicSpline(Tt, pt)(T_fine)/p_fine - 1)
e_cla = np.abs(spline_clamped(Tt, pt, T_fine, dpdT(Tt[0]), dpdT(Tt[-1]))/p_fine - 1)
ends = (T_fine < 20) | (T_fine > 90)
print(f"max error near the ends: natural {e_nat[ends].max():.2e}, clamped {e_cla[ends].max():.2e}")
print(f"max error in the middle: natural {e_nat[~ends].max():.2e}, clamped {e_cla[~ends].max():.2e}")

max error near the ends: natural 1.50e-02, clamped 1.61e-04
max error in the middle: natural 2.38e-03, clamped 1.25e-04


With the correct end slopes the error near the ends of the table drops dramatically, and the whole
interpolant improves. The first and last rows are the only change: they replace "zero curvature" by the
slope condition obtained by differentiating the end cubics.